In [1]:
import numpy as np
import polars as pl
import pandas as pd
import tensorflow as tf
from PyEMD import EMD, CEEMDAN, EEMD
from pyeemd import ceemdan
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from working_data import clean_cols, clean_non_minute_rows, alt_label_df as label_df, normalize_by_window, split_df


2024-10-20 19:44:30.183736: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-20 19:44:30.218853: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-20 19:44:31.963442: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
NORMALIZING_WINDOW_SIZE = 180
LABELING_WINDOW_SIZE = 20
POSITIVE_SLOPE = 0.3
LABEL_CUR_CANDLE_MULTIPLIER = 0
LABEL_MEAN_MULTIPLIER = 8
BATCH_SIZE = 32
NUM_IMFS = 6
NUM_TOKENS = 240

In [3]:
source_csv = "data/GBPUSD/minutes.csv"
working_path = "working"

In [4]:
df = pd.read_csv(source_csv)
df = clean_non_minute_rows(df)
df = clean_cols(df)
break_point = len(df) - len(df)//20
df = df[break_point:]
df = normalize_by_window(
    df, 
    window_size=NORMALIZING_WINDOW_SIZE, 
    normalizing_cols=[
        'open',
        'high',
        'low',
        'close',
    ])
print("labeling")
df = label_df(df, window_size=LABELING_WINDOW_SIZE, mean_multiplier=LABEL_MEAN_MULTIPLIER, cur_candle_multiplier=LABEL_CUR_CANDLE_MULTIPLIER)
print(df['target'].value_counts())
split_df(
    df=df, 
    dump_path=working_path, 
    cols=[
        'close',
        'target'
    ])

labeling
target
0    426393
1     22541
Name: count, dtype: int64


In [5]:
def prepare_data(file_path, num_tokens, window_size=1440, batch_size=32, num_imfs=8, smote=False, shuffle=False, col='close'):
    scaler = StandardScaler()
    emd_range = window_size
    # Load CSV lazily with Polars
    df_lazy = pl.scan_csv(file_path).select([col, 'target'])
    
    # Collect the dataframe and determine total number of rows
    df_collected = df_lazy.collect()
    total_rows = df_collected.shape[0]
    if smote:
        df_collected = df_collected.with_columns(pl.arange(0, total_rows).alias("index"))
        indices_target_1 = df_collected.filter(
            (pl.col("target") == 1) & (pl.col("index") >= num_tokens)
        ).select("index").to_series().to_list()

        # Get indices where target is 0 and >= num_tokens
        indices_target_0 = df_collected.filter(
            (pl.col("target") == 0) & (pl.col("index") >= num_tokens)
        ).select("index").to_series().to_list()

        df_collected = df_collected.drop('index')
    
    while True:  # Loop to reshuffle and restart at each epoch
        # Create an array of indices to use for shuffling
        indices = list(range(emd_range, total_rows))
        if smote:
            indices_target_1_complete = []
            while len(indices_target_1_complete) < len(indices_target_0):
                indices_target_1_complete += indices_target_1

            indices_target_1_complete = indices_target_1_complete[:len(indices_target_0)]

            indices = indices_target_0 + indices_target_1_complete

        # Shuffle indices if required
        if shuffle:
            np.random.shuffle(indices)

        input_lists = [[] for _ in range(num_imfs)]
        target_list = []

        for idx in indices:
            #create imfs
            signal = np.array(df_collected[col][idx - emd_range + 1:idx + 1])
            signal = scaler.fit_transform(signal.reshape(-1, 1)).flatten()
            imfs = ceemdan(signal, num_imfs=num_imfs)

            # Check the number of IMFs generated
            cur_num_imfs = imfs.shape[0]

            # If fewer than 8 IMFs are generated, pad with flat signals (zeros)
            if cur_num_imfs < num_imfs:
                # Create a flat signal (array of zeros) with the same length as the original signal
                flat_signal = np.zeros_like(signal)                
                # Append the required number of flat signals to imfs
                imfs = np.vstack([imfs] + [flat_signal] * (num_imfs - cur_num_imfs))

            # Fetch the previous `num_prev + 1` rows for the input based on the current index
            input_rows = imfs[:, -num_tokens:]  # Exclude 'target' for input
            target_value = df_collected[idx, -1]  # Get 'target' for the target


            for input_idx in range(num_imfs):
                input_lists[input_idx].append(input_rows[input_idx])
            target_list.append(target_value)

            print(idx)

            # Yield once we have enough for a batch
            if len(input_lists[0]) == batch_size:
                # Convert lists to NumPy arrays
                input_arrays = [np.array(input_list) for input_list in input_lists]
                target_array = np.array(target_list)

                # Convert NumPy arrays to TensorFlow tensors
                input_tensors = [tf.reshape(tf.convert_to_tensor(input_array, dtype=tf.float32), (batch_size, num_tokens, 1)) for input_array in input_arrays]
                target_tensor = tf.convert_to_tensor(target_array, dtype=tf.int32)

                yield tuple(input_tensors), target_tensor

                # Reset lists for the next batch
                input_lists = [[] for _ in range(num_imfs)]
                target_list.clear()
        
        break  # Uncomment if you want to stop after one full pass

In [6]:
def create_dataset_generator(file_path, batch_size, num_tokens, window_size=1440, num_imfs=10, shuffle=False, repeat=False, smote=False, col='close'):
    dataset = tf.data.Dataset.from_generator(
        lambda: prepare_data(file_path, window_size=window_size, batch_size=batch_size, num_imfs=num_imfs, num_tokens=num_tokens, shuffle=shuffle, smote=smote, col=col),
        output_signature=(
            tuple([tf.TensorSpec(shape=(None, num_tokens, 1), dtype=tf.float32) for _ in range(num_imfs)]),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    )
    if repeat:
        dataset = dataset.repeat()
    return dataset 


In [7]:
train_dataset = create_dataset_generator('working/train.csv', batch_size=BATCH_SIZE, repeat=True, num_tokens=NUM_TOKENS, shuffle=True, num_imfs=NUM_IMFS, col='close')
val_dataset = create_dataset_generator('working/val.csv', batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, num_imfs=NUM_IMFS, col='close')
test_dataset = create_dataset_generator('working/test.csv', batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, num_imfs=NUM_IMFS, col='close')

2024-10-20 19:46:02.483366: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-20 19:46:02.521032: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-20 19:46:02.521088: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-20 19:46:02.523982: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-20 19:46:02.524036: I external/local_xla/xla/stream_executor

In [8]:
for data in train_dataset.take(1):
    inputs, labels = data
    print(len(inputs))

97754
32932
293404
22801
48929
211375
83305
295890
259741
195993
301685
56380
121972
292380
237334
303435
275850
85116
27613
285924
224446
203325
20614
41456
269661
311697
54022
313915
91859
72643
86336
290839
6


2024-10-20 19:46:16.001049: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
inputs[0][4]

<tf.Tensor: shape=(240, 1), dtype=float32, numpy=
array([[-4.44021495e-03],
       [ 5.40038049e-02],
       [-3.06368731e-02],
       [ 4.24828473e-03],
       [-1.68980099e-02],
       [-9.72887129e-03],
       [ 5.37591428e-02],
       [-5.30790985e-02],
       [ 4.67881411e-02],
       [ 5.20550534e-02],
       [-2.60099303e-02],
       [-2.50995427e-01],
       [ 2.80845553e-01],
       [-2.34160274e-02],
       [-2.03539491e-01],
       [ 4.52341065e-02],
       [ 7.61111900e-02],
       [-1.71081983e-02],
       [ 2.27141827e-02],
       [-7.50041828e-02],
       [ 1.03520721e-01],
       [-7.03701973e-02],
       [ 1.78611502e-02],
       [ 2.95415148e-02],
       [-4.69057858e-02],
       [ 3.62957418e-02],
       [ 8.77681561e-03],
       [ 1.54748941e-02],
       [-3.82118188e-02],
       [ 1.25062214e-02],
       [ 8.96929502e-02],
       [-9.80316848e-02],
       [ 5.61730983e-03],
       [ 1.22758653e-02],
       [-1.31436419e-02],
       [-1.26416758e-02],
       [ 2.980